In [2]:
import sys
import os

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.nlp.news_fetcher import NewsFetcher
from src.nlp.sentiment_analyzer import SentimentAnalyzer
from src.data.spark_pipeline import get_spark_session
from src.data.databricks_client import DatabricksClient

print("Phase 2 imports successful!")

Phase 2 imports successful!


In [3]:
# 1. Fetch news articles
fetcher = NewsFetcher()
news_df = fetcher.fetch_ticker_news("AAPL", limit=10)

# 2. Analyze sentiment
analyzer = SentimentAnalyzer()
sentiment_df = analyzer.add_sentiment_features(news_df)

sentiment_df[["timestamp", "ticker", "title", "compound_score", "pos_score", "neg_score"]].head()

[2026-09-22 03:32:48] [INFO] [stonks_maker]: Fetching news for AAPL via yfinance fallback...
[2026-09-22 03:32:49] [INFO] [stonks_maker]: Computing sentiment scores for news dataset...
[2026-09-22 03:32:49] [INFO] [stonks_maker]: Sentiment scoring completed.


,timestamp,ticker,title,compound_score,pos_score,neg_score
0,2026-09-21 19:52:00+00:00,AAPL,Meta stock jumps as Wells Fargo raises price t...,0.7906,0.134,0.016
1,2026-09-21 17:41:00+00:00,AAPL,Apple’s new iPhone Duo: Will consumers bite?,0.7269,0.103,0.000
2,2026-09-22 03:21:33+00:00,AAPL,META Stock Rallies As Muse AI Tops App Charts ...,0.6705,0.147,0.000
3,2026-09-22 03:02:39+00:00,AAPL,Apple Launches New Products in Time for New CEO,0.0000,0.000,0.000
4,2026-09-22 02:35:28+00:00,AAPL,How Tim Cook Set Up Apple Stock's Next Growth ...,0.6369,0.144,0.000


In [4]:
# Convert sentiment dataset to Spark DataFrame
spark = get_spark_session()
spark_news_df = spark.createDataFrame(sentiment_df)

# Save to Parquet/Delta storage
db_client = DatabricksClient()
db_client.write_dataset(spark_news_df, table_name="aapl_sentiment")

print("Phase 2 Execution Complete!")

[2026-09-22 03:33:10] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-22 03:33:15] [INFO] [stonks_maker]: Successfully saved to data/processed/aapl_sentiment
Phase 2 Execution Complete!
